In [ ]:
# Notebook: governance_setup
# Purpose: Creates the LGPD dynamic masking function (regra_mascaramento) in Unity Catalog and
#          publishes the governance policy (politica_governanca) as a task value.

import json
import os
from databricks.sdk.runtime import spark
from databricks.sdk.runtime import dbutils
from grupo10.utils.logger import get_logger


logger = get_logger("governance_setup")

# Resolves the notebook path dynamically so the JSON is found in the Workspace
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = os.path.dirname(notebook_path)
if not notebook_dir.startswith("/Workspace"):
    notebook_dir = "/Workspace" + notebook_dir

governanca_config_path = os.path.join(notebook_dir, "governance_config.json")

with open(governanca_config_path, "r", encoding="utf-8") as f:
    politica_governanca = json.load(f)

# Ensures the masking function exists in Unity Catalog before the loop
spark.sql("""
CREATE OR REPLACE FUNCTION lh_nautical.bronze.regra_mascaramento(valor STRING)
RETURN CASE
  WHEN is_account_group_member('admins') THEN valor
  WHEN is_account_group_member('engenharia_dados') THEN valor
  ELSE '████████ (RESTRITO LGPD)'
END;
""")
logger.info("Dynamic masking function 'regra_mascaramento' ensured in Unity Catalog.")

#Publishes the governance policy as a task value
dbutils.jobs.taskValues.set(key="politica_governanca", value=politica_governanca)
logger.info("Published 'politica_governanca' as task value.")
